In [26]:
!pip install ultralytics opencv-python-headless numpy scipy

In [27]:
import cv2
import numpy as np
import os
import sys
from pathlib import Path
from collections import defaultdict, deque
from ultralytics import YOLO
from scipy.ndimage import gaussian_filter

YOLO_AVAILABLE = True
SCIPY_AVAILABLE = True

In [28]:
class Config:
    YOLO_MODEL         = "yolov8n.pt"
    CONF_THRESHOLD     = 0.40
    PERSON_CLASS_ID    = 0
    FRAME_SKIP         = 2
    INPUT_RESIZE_W     = 640
    PITCH_W            = 800
    PITCH_H            = 550
    PITCH_MARGIN       = 40
    HEATMAP_WINDOW     = 60
    HEATMAP_SIGMA      = 18
    HEATMAP_ALPHA      = 0.65
    TRACK_HISTORY      = 30
    TEAM_A_HSV_LOWER   = np.array([0,  80, 80])
    TEAM_A_HSV_UPPER   = np.array([15, 255, 255])
    TEAM_B_HSV_LOWER   = np.array([100, 80, 80])
    TEAM_B_HSV_UPPER   = np.array([130, 255, 255])
    OUTPUT_FPS         = 25

In [29]:
class PitchRenderer:
    def __init__(self, w=Config.PITCH_W, h=Config.PITCH_H, margin=Config.PITCH_MARGIN):
        self.W = w
        self.H = h
        self.M = margin
        self.x0 = margin
        self.y0 = margin
        self.x1 = w - margin
        self.y1 = h - margin
        self.pw = self.x1 - self.x0
        self.ph = self.y1 - self.y0
        self._base = self._draw_base()

    def _draw_base(self):
        img = np.zeros((self.H, self.W, 3), dtype=np.uint8)
        img[:] = (34, 139, 34)
        lc, lt = (255, 255, 255), 2
        x0, y0, x1, y1 = self.x0, self.y0, self.x1, self.y1
        pw, ph = self.pw, self.ph

        cv2.rectangle(img, (x0, y0), (x1, y1), lc, lt)
        cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
        cv2.line(img, (cx, y0), (cx, y1), lc, lt)
        cv2.circle(img, (cx, cy), int(ph * 0.15), lc, lt)
        cv2.circle(img, (cx, cy), 3, lc, -1)

        pa_w = int(pw * 0.16)
        pa_h = int(ph * 0.59)
        pa_dy = (ph - pa_h) // 2
        cv2.rectangle(img, (x0, y0 + pa_dy), (x0 + pa_w, y1 - pa_dy), lc, lt)
        cv2.rectangle(img, (x1 - pa_w, y0 + pa_dy), (x1, y1 - pa_dy), lc, lt)

        ga_w = int(pw * 0.053)
        ga_h = int(ph * 0.27)
        ga_dy = (ph - ga_h) // 2
        cv2.rectangle(img, (x0, y0 + ga_dy), (x0 + ga_w, y1 - ga_dy), lc, lt)
        cv2.rectangle(img, (x1 - ga_w, y0 + ga_dy), (x1, y1 - ga_dy), lc, lt)

        goal_d = int(ph * 0.11)
        goal_dy = (ph - goal_d) // 2
        cv2.rectangle(img, (x0 - 8, y0 + goal_dy), (x0, y1 - goal_dy), (200, 200, 200), 2)
        cv2.rectangle(img, (x1, y0 + goal_dy), (x1 + 8, y1 - goal_dy), (200, 200, 200), 2)

        ps_x = int(pw * 0.108)
        cv2.circle(img, (x0 + ps_x, cy), 3, lc, -1)
        cv2.circle(img, (x1 - ps_x, cy), 3, lc, -1)

        r = int(ph * 0.03)
        for corner in [(x0, y0), (x1, y0), (x0, y1), (x1, y1)]:
            cv2.ellipse(img, corner, (r, r), 0, 0, 360, lc, lt)

        return img

    def get_blank(self):
        return self._base.copy()

    def get_corners(self):
        return np.float32([
            [self.x0, self.y0],
            [self.x1, self.y0],
            [self.x1, self.y1],
            [self.x0, self.y1]
        ])

In [30]:
class HomographyManager:
    def __init__(self, pitch):
        self.pitch  = pitch
        self.H_mat  = None
        self.src_pts = None

    def set_from_points(self, src_pts):
        dst_pts = self.pitch.get_corners()
        self.src_pts = src_pts
        self.H_mat, status = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
        if self.H_mat is None:
            raise ValueError("Homography computation failed — check input points")
        print(f"[INFO] Homography computed (inliers: {status.sum()}/4)")

    def transform_point(self, px, py):
        if self.H_mat is None:
            return None
        pt = np.array([[[px, py]]], dtype=np.float32)
        out = cv2.perspectiveTransform(pt, self.H_mat)
        return out[0][0]

    def transform_points(self, pts):
        if self.H_mat is None or len(pts) == 0:
            return np.array([])
        src = pts.reshape(-1, 1, 2).astype(np.float32)
        dst = cv2.perspectiveTransform(src, self.H_mat)
        return dst.reshape(-1, 2)

    def is_inside_pitch(self, x, y):
        m = self.pitch.M
        return (m <= x <= self.pitch.W - m) and (m <= y <= self.pitch.H - m)

In [31]:
class TeamClassifier:
    def classify(self, frame, bbox):
        x1, y1, x2, y2 = [int(v) for v in bbox]
        h = y2 - y1
        crop = frame[y1:y1 + h//3, x1:x2]
        if crop.size == 0:
            return 2

        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        mask_a = cv2.inRange(hsv, Config.TEAM_A_HSV_LOWER, Config.TEAM_A_HSV_UPPER)
        mask_b = cv2.inRange(hsv, Config.TEAM_B_HSV_LOWER, Config.TEAM_B_HSV_UPPER)

        score_a = np.count_nonzero(mask_a)
        score_b = np.count_nonzero(mask_b)

        if max(score_a, score_b) < 20:
            return 2
        return 0 if score_a >= score_b else 1

In [32]:
class HeatmapBuilder:
    def __init__(self, w, h, window=Config.HEATMAP_WINDOW, sigma=Config.HEATMAP_SIGMA):
        self.W = w
        self.H = h
        self.window = window
        self.sigma  = sigma
        self.history = deque(maxlen=window)

    def add_frame(self, pts):
        self.history.append(list(pts))

    def build_map(self):
        acc = np.zeros((self.H, self.W), dtype=np.float32)
        for frame_pts in self.history:
            for x, y in frame_pts:
                xi = int(np.clip(x, 0, self.W-1))
                yi = int(np.clip(y, 0, self.H-1))
                acc[yi, xi] += 1.0

        if SCIPY_AVAILABLE:
            acc = gaussian_filter(acc, sigma=self.sigma)
        else:
            k = int(self.sigma * 3) | 1
            acc = cv2.GaussianBlur(acc, (k, k), self.sigma)

        max_val = acc.max()
        if max_val > 0:
            acc /= max_val
        return acc

    def colorize(self, hmap):
        uint8 = (hmap * 255).astype(np.uint8)
        return cv2.applyColorMap(uint8, cv2.COLORMAP_JET)

In [33]:
class SimpleTracker:
    def __init__(self, max_dist=80.0):
        self.tracks  = {}
        self.next_id = 1
        self.max_dist = max_dist

    def update(self, detections):
        if not detections:
            for tid in list(self.tracks):
                self.tracks[tid]["lost"] += 1
                if self.tracks[tid]["lost"] > 10:
                    del self.tracks[tid]
            return []

        new_centers = [((d[0]+d[2])/2, (d[1]+d[3])/2) for d in detections]
        assigned    = {}
        used_tracks = set()

        for ni, nc in enumerate(new_centers):
            best_id, best_d = None, self.max_dist
            for tid, tk in self.tracks.items():
                if tid in used_tracks:
                    continue
                tc = tk["center"]
                d  = np.hypot(nc[0]-tc[0], nc[1]-tc[1])
                if d < best_d:
                    best_d, best_id = d, tid
            if best_id is not None:
                assigned[ni] = best_id
                used_tracks.add(best_id)

        result = []
        for ni, det in enumerate(detections):
            tid = assigned[ni] if ni in assigned else self.next_id
            if ni not in assigned:
                self.next_id += 1
            self.tracks[tid] = {"center": new_centers[ni], "lost": 0}
            result.append((*det, tid))

        for tid in list(self.tracks):
            if tid not in used_tracks and tid not in assigned.values():
                self.tracks[tid]["lost"] += 1
                if self.tracks[tid]["lost"] > 10:
                    del self.tracks[tid]

        return result

In [34]:
class FootballAnalyser:
    def __init__(self, video_path, output_dir="output", manual_corners=None):
        self.video_path     = video_path
        self.output_dir     = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.manual_corners = manual_corners

        self.pitch      = PitchRenderer()
        self.homo       = HomographyManager(self.pitch)
        self.classifier = TeamClassifier()
        self.heatmap    = HeatmapBuilder(self.pitch.W, self.pitch.H)
        self.tracker    = SimpleTracker()
        self.track_history = defaultdict(lambda: deque(maxlen=Config.TRACK_HISTORY))

        if YOLO_AVAILABLE:
            print(f"[INFO] Loading YOLO model: {Config.YOLO_MODEL}")
            self.model = YOLO(Config.YOLO_MODEL)
        else:
            self.model = None
            print("[ERROR] YOLO not available — detections will be empty")

    def _select_corners_manually(self, frame):
        # NOTE: cv2.imshow() does NOT work in Colab.
        # Use the manual_corners argument instead (see Cell 10).
        pts = []
        clone = frame.copy()

        def click(event, x, y, flags, param):
            if event == cv2.EVENT_LBUTTONDOWN and len(pts) < 4:
                pts.append([x, y])

        cv2.imshow("Select 4 corners: TL→TR→BR→BL", clone)
        cv2.setMouseCallback("Select 4 corners: TL→TR→BR→BL", click)
        while len(pts) < 4:
            cv2.waitKey(20)
        cv2.destroyAllWindows()
        return np.float32(pts)

    def _try_auto_corners(self, frame):
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur  = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)
        lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100,
                                minLineLength=100, maxLineGap=20)
        if lines is None or len(lines) < 4:
            print("[INFO] Auto corner detection failed — switching to manual")
            return None

        all_pts = lines.reshape(-1, 2)
        hull    = cv2.convexHull(all_pts.astype(np.float32))
        if hull.shape[0] < 4:
            return None

        epsilon = 0.05 * cv2.arcLength(hull, True)
        approx  = cv2.approxPolyDP(hull, epsilon, True)
        if len(approx) == 4:
            pts  = approx.reshape(4, 2).astype(np.float32)
            rect = np.zeros((4, 2), dtype=np.float32)
            s    = pts.sum(axis=1)
            diff = np.diff(pts, axis=1)
            rect[0] = pts[np.argmin(s)]
            rect[2] = pts[np.argmax(s)]
            rect[1] = pts[np.argmin(diff)]
            rect[3] = pts[np.argmax(diff)]
            print("[INFO] Auto corners detected")
            return rect

        print("[INFO] Auto corner shape not 4-sided — switching to manual")
        return None

    def _detect_players(self, frame):
        if self.model is None:
            return []
        h, w = frame.shape[:2]
        scale = Config.INPUT_RESIZE_W / w
        small = cv2.resize(frame, (Config.INPUT_RESIZE_W, int(h * scale)))
        results = self.model(small, conf=Config.CONF_THRESHOLD,
                             classes=[Config.PERSON_CLASS_ID], verbose=False)
        detections = []
        inv_scale  = 1.0 / scale
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            x1, x2 = x1 * inv_scale, x2 * inv_scale
            y1, y2 = y1 * inv_scale, y2 * inv_scale
            team = self.classifier.classify(frame, (x1, y1, x2, y2))
            detections.append((x1, y1, x2, y2, conf, team))
        return detections

    def _draw_detections_on_frame(self, frame, tracked):
        team_colors = [(0, 60, 220), (220, 80, 0), (128, 128, 128)]
        out = frame.copy()
        for (x1,y1,x2,y2,conf,team,tid) in tracked:
            c = team_colors[team]
            cv2.rectangle(out, (int(x1),int(y1)), (int(x2),int(y2)), c, 2)
            label = f"#{tid} {conf:.2f}"
            cv2.putText(out, label, (int(x1), int(y1)-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 1)
        return out

    def _draw_topview(self, tracked_pitch_pts, heatmap_img):
        pitch = self.pitch.get_blank()
        hmask = cv2.cvtColor(heatmap_img, cv2.COLOR_BGR2GRAY)
        _, hmask = cv2.threshold(hmask, 10, 255, cv2.THRESH_BINARY)
        pitch = cv2.addWeighted(pitch, 1 - Config.HEATMAP_ALPHA,
                                heatmap_img, Config.HEATMAP_ALPHA, 0, dst=pitch)
        pitch_base = self.pitch.get_blank()
        pitch = np.where(hmask[:, :, np.newaxis] > 0, pitch, pitch_base)

        for tid, trail in self.track_history.items():
            pts = list(trail)
            for i in range(1, len(pts)):
                alpha = i / len(pts)
                col = (int(255*alpha), int(255*alpha), int(255*alpha))
                cv2.line(pitch,
                         (int(pts[i-1][0]), int(pts[i-1][1])),
                         (int(pts[i][0]),   int(pts[i][1])), col, 1)

        team_dot_colors = [(30, 60, 255), (255, 80, 30), (200, 200, 200)]
        for (px, py, team, tid) in tracked_pitch_pts:
            c = team_dot_colors[team]
            cv2.circle(pitch, (int(px), int(py)), 7, c, -1)
            cv2.circle(pitch, (int(px), int(py)), 7, (255,255,255), 1)
            cv2.putText(pitch, str(tid), (int(px)+8, int(py)-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255,255,255), 1)
        return pitch

    def run(self):
        cap = cv2.VideoCapture(self.video_path)
        if not cap.isOpened():
            print(f"[ERROR] Cannot open video: {self.video_path}")
            return

        fps    = cap.get(cv2.CAP_PROP_FPS) or Config.OUTPUT_FPS
        total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out_fps = fps / Config.FRAME_SKIP
        print(f"[INFO] Video: {w_orig}x{h_orig} @ {fps:.1f}fps, {total} frames")

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        combined_out = cv2.VideoWriter(
            str(self.output_dir / "overlay_heatmap.mp4"), fourcc, out_fps,
            (w_orig + self.pitch.W, max(h_orig, self.pitch.H)))
        topview_out = cv2.VideoWriter(
            str(self.output_dir / "topview_radar.mp4"), fourcc, out_fps,
            (self.pitch.W, self.pitch.H))

        ret, first_frame = cap.read()
        if not ret:
            print("[ERROR] Empty video")
            return
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

        if self.manual_corners is not None:
            src_pts = np.float32(self.manual_corners)
        else:
            src_pts = self._try_auto_corners(first_frame)
            if src_pts is None:
                src_pts = self._select_corners_manually(first_frame)

        self.homo.set_from_points(src_pts)

        frame_idx = 0
        processed  = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1
            if frame_idx % Config.FRAME_SKIP != 0:
                continue

            processed += 1
            if processed % 50 == 0:
                print(f"[INFO] Frame {frame_idx}/{total} ({100*frame_idx//total}%)")

            detections = self._detect_players(frame)
            tracked    = self.tracker.update(detections)

            foot_pts      = []
            pitch_pts_all = []
            for (x1,y1,x2,y2,conf,team,tid) in tracked:
                fx = (x1 + x2) / 2
                fy = y2
                foot_pts.append((fx, fy))
                tp = self.homo.transform_point(fx, fy)
                if tp is not None and self.homo.is_inside_pitch(tp[0], tp[1]):
                    pitch_pts_all.append((tp[0], tp[1], team, tid))
                    self.track_history[tid].append((tp[0], tp[1]))

            hm_pts    = [(p[0], p[1]) for p in pitch_pts_all]
            self.heatmap.add_frame(hm_pts)

            hmap      = self.heatmap.build_map()
            hmap_bgr  = self.heatmap.colorize(hmap)

            overlay_frame = self._draw_detections_on_frame(frame, tracked)
            topview_frame = self._draw_topview(pitch_pts_all, hmap_bgr)

            oh, ow = overlay_frame.shape[:2]
            ph, pw = self.pitch.H, self.pitch.W
            if oh > ph:
                pad = np.zeros((oh - ph, pw, 3), dtype=np.uint8)
                topview_frame = np.vstack([topview_frame, pad])
            elif ph > oh:
                pad = np.zeros((ph - oh, ow, 3), dtype=np.uint8)
                overlay_frame = np.vstack([overlay_frame, pad])

            combined = np.hstack([overlay_frame, topview_frame])
            combined_out.write(combined)
            topview_out.write(topview_frame)

        cap.release()
        combined_out.release()
        topview_out.release()
        print(f"\n[DONE] Processed {processed} frames")
        print(f"  → overlay_heatmap.mp4 ({w_orig + self.pitch.W}x{max(h_orig, self.pitch.H)})")
        print(f"  → topview_radar.mp4   ({self.pitch.W}x{self.pitch.H})")

In [35]:
import urllib.request

video_url = "https://raw.githubusercontent.com/sunayana90/virtual-ad-overlay-homography-yolo/main/football.mp4"
video_filename = "football.mp4"

print("Downloading video...")
urllib.request.urlretrieve(video_url, video_filename)
print(f"Downloaded: {video_filename}")

# Provide 4 pitch corner coordinates (TL, TR, BR, BL)
manual_corners = [
    [100, 50],   # Top-Left
    [1180, 50],  # Top-Right
    [1180, 670], # Bottom-Right
    [100, 670],  # Bottom-Left
]

analyser = FootballAnalyser(
    video_path=video_filename,
    output_dir="output",
    manual_corners=manual_corners
)
analyser.run()

Downloaded: football.mp4
[INFO] Loading YOLO model: yolov8n.pt
[INFO] Video: 1920x1080 @ 25.0fps, 750 frames
[INFO] Homography computed (inliers: 4/4)
[INFO] Frame 100/750 (13%)
[INFO] Frame 200/750 (26%)
[INFO] Frame 300/750 (40%)
[INFO] Frame 400/750 (53%)
[INFO] Frame 500/750 (66%)
[INFO] Frame 600/750 (80%)
[INFO] Frame 700/750 (93%)

[DONE] Processed 375 frames
  → overlay_heatmap.mp4 (2720x1080)
  → topview_radar.mp4   (800x550)


In [36]:
import cv2
from google.colab.patches import cv2_imshow
cap = cv2.VideoCapture("football.mp4")
ret, frame = cap.read()
cap.release()
print("Frame size:", frame.shape)  # (height, width, channels)
cv2_imshow(frame)

Output hidden; open in https://colab.research.google.com to view.

In [37]:
from google.colab import files
files.download("output/overlay_heatmap.mp4")
files.download("output/topview_radar.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>